# Simple Fused ONNX Script Quickstart

This notebook shows a minimal flow:

1. Convert BERT classifier to ONNX using CHiRPE script
2. Build fused tokenizer+classifier ONNX using CHiRPE script
3. Run fused model inference from raw text

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import onnxruntime as ort
from onnxruntime_extensions import get_library_path

2026-04-23 11:19:39.670521646 [W:onnxruntime:Default, device_discovery.cc:325 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:92 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from repo root or notebooks/.")

BACKBONE = "bert"
VERSION = 1
OPSET = 18
MAX_LENGTH = 128

CHECKPOINT_DIR = REPO_ROOT / "outputs" / "string_onnx_checkpoints" / BACKBONE / "best_model"
CLASSIFIER_REPO = REPO_ROOT / "outputs" / "string_onnx_baseline_classifier"
FUSED_REPO = REPO_ROOT / "outputs" / "string_onnx_fused"
REPORT_ROOT = REPO_ROOT / "reports" / "fused_smoke"

BASELINE_MODEL_NAME = f"chirpe_{BACKBONE}_baseline"
FUSED_MODEL_NAME = f"chirpe_{BACKBONE}_string"

if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError("Checkpoint directory is missing. Generate quick checkpoints first.")

print(f"Backbone: {BACKBONE}")
print(f"Baseline model name: {BASELINE_MODEL_NAME}")
print(f"Fused model name: {FUSED_MODEL_NAME}")

Backbone: bert
Baseline model name: chirpe_bert_baseline
Fused model name: chirpe_bert_string


In [3]:
export_cmd = [
    sys.executable,
    "scripts/onnx/export_triton_onnx.py",
    "--model-dir",
    str(CHECKPOINT_DIR),
    "--triton-repo",
    str(CLASSIFIER_REPO),
    "--model-name",
    BASELINE_MODEL_NAME,
    "--version",
    str(VERSION),
    "--opset",
    str(OPSET),
    "--max-length",
    str(MAX_LENGTH),
]

print("Running classifier export script...")
result = subprocess.run(
    export_cmd,
    cwd=str(REPO_ROOT),
    check=True,
    text=True,
    capture_output=True,
)

print("Classifier export done.")

Running classifier export script...


Classifier export done.


In [4]:
build_cmd = [
    sys.executable,
    "scripts/onnx/build_fused_string_onnx.py",
    "--checkpoint-root",
    str(REPO_ROOT / "outputs" / "string_onnx_checkpoints"),
    "--classifier-repo",
    str(CLASSIFIER_REPO),
    "--output-repo",
    str(FUSED_REPO),
    "--report-root",
    str(REPORT_ROOT),
    "--backbones",
    BACKBONE,
    "--classifier-suffix",
    "baseline",
    "--fused-suffix",
    "string",
    "--version",
    str(VERSION),
]

print("Running fused build script...")
result = subprocess.run(
    build_cmd,
    cwd=str(REPO_ROOT),
    check=True,
    text=True,
    capture_output=True,
)

print("Fused build done.")

Running fused build script...


Fused build done.


In [5]:
fused_model_path = FUSED_REPO / FUSED_MODEL_NAME / str(VERSION) / "model.onnx"
fused_metadata_path = FUSED_REPO / FUSED_MODEL_NAME / str(VERSION) / "export_metadata.json"

if not fused_model_path.exists():
    raise FileNotFoundError("Fused model was not created.")
if not fused_metadata_path.exists():
    raise FileNotFoundError("Fused metadata was not created.")

with open(fused_metadata_path, "r") as file:
    fused_metadata = json.load(file)

print("Artifacts ready.")
print("Fused metadata summary:")
print(json.dumps({
    "backbone": fused_metadata.get("backbone"),
    "fused_model_name": fused_metadata.get("fused_model_name"),
    "inputs": fused_metadata.get("inputs", []),
    "outputs": fused_metadata.get("outputs", []),
}, indent=2))

Artifacts ready.
Fused metadata summary:
{
  "backbone": "bert",
  "fused_model_name": "chirpe_bert_string",
  "inputs": [
    {
      "name": "text",
      "type": "tensor(string)",
      "shape": [
        null
      ]
    }
  ],
  "outputs": [
    {
      "name": "logits",
      "type": "tensor(float)",
      "shape": [
        "batch_size",
        2
      ]
    }
  ]
}


In [6]:
ortx_library = Path(get_library_path())
if not ortx_library.exists():
    raise FileNotFoundError("ORT Extensions shared library was not found.")

session_options = ort.SessionOptions()
session_options.register_custom_ops_library(str(ortx_library))
session = ort.InferenceSession(str(fused_model_path), session_options, providers=["CPUExecutionProvider"])

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("Fused ONNX session loaded.")
print("Inputs:", [{"name": i.name, "type": i.type, "shape": i.shape} for i in session.get_inputs()])
print("Outputs:", [{"name": o.name, "type": o.type, "shape": o.shape} for o in session.get_outputs()])

Fused ONNX session loaded.
Inputs: [{'name': 'text', 'type': 'tensor(string)', 'shape': [None]}]
Outputs: [{'name': 'logits', 'type': 'tensor(float)', 'shape': ['batch_size', 2]}]


In [7]:
label_map = {0: "Healthy", 1: "CHR-P"}

def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / exp_vals.sum(axis=-1, keepdims=True)

def predict_fused(text: str) -> dict:
    # single-text call (string input)
    logits = session.run([output_name], {input_name: np.array([text], dtype=object)})[0]
    probs = softmax(logits)
    pred = int(np.argmax(probs, axis=-1)[0])
    return {
        "text": text,
        "prediction": pred,
        "label": label_map.get(pred, str(pred)),
        "confidence": float(probs[0, pred]),
        "probabilities": probs[0].tolist(),
    }

examples = [
    "I feel mostly fine and my routine is stable.",
    "Sometimes I hear whispers and feel people can read my thoughts.",
    "I occasionally feel confused and suspicious in public places.",
]

for idx, text in enumerate(examples, start=1):
    result = predict_fused(text)
    print(f"Text #{idx}: {result['text']}")
    print(f"  Prediction: {result['label']} ({result['prediction']})")
    print(f"  Confidence: {result['confidence']:.4f}")
    print(f"  Probabilities: {[round(v, 4) for v in result['probabilities']]}")
    print()

Text #1: I feel mostly fine and my routine is stable.
  Prediction: CHR-P (1)
  Confidence: 0.5928
  Probabilities: [0.4072, 0.5928]

Text #2: Sometimes I hear whispers and feel people can read my thoughts.
  Prediction: CHR-P (1)
  Confidence: 0.6664
  Probabilities: [0.3336, 0.6664]

Text #3: I occasionally feel confused and suspicious in public places.
  Prediction: CHR-P (1)
  Confidence: 0.5809
  Probabilities: [0.4191, 0.5809]

